# Gaussian Point-Cloud & Camera-Route Visualization (WebGL)

**Why this notebook looks different from a typical Jupyter viz notebook**: an earlier
version embedded interactive Plotly 3D scatters directly in the `.ipynb` file, which
made it ~10MB and slow to open/diff (every point's position+color serialized as JSON,
baked into the notebook itself). This version instead:

1. Exports point-cloud and camera data to compact **binary files** (`viz/data/*.bin`
   — raw `Float32`/`Uint8` arrays, no JSON/text overhead) via the code cells below.
2. Writes a small, self-contained **Three.js WebGL viewer** (`viz/index.html`) that
   fetches those binary files directly in the browser.
3. Starts a plain local HTTP server (Python's built-in `http.server`, no extra
   dependency) and gives you a `http://localhost:8000` link to open.

The notebook itself stays lightweight (just the export code); all the actual
rendering happens in your browser's GPU via WebGL, which comfortably handles far more
points than an embedded Plotly figure could.

**What's in the viewer:**
- The **initial** point cloud (COLMAP's sparse SfM reconstruction, before any Gaussian
  training).
- The **final** trained Gaussian centers (after 30,000 iterations).
- A **densification-gap** layer: a voxel-grid density-ratio comparison between initial
  and final (see §3 for why this is computed as a voxel ratio rather than a naive
  per-point "is this Gaussian new?" classifier — the naive version turned out to be
  uninformative).
- The **camera route**: COLMAP's recovered camera positions and viewing directions for
  every training photo, connected in capture-index order, with small wireframe frustums
  showing where each camera was looking.

## Config

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
from plyfile import PlyData
from scipy.spatial import cKDTree
import matplotlib as mpl
import matplotlib.colors as mcolors

REPO_ROOT = Path.cwd()
RESULTS_ROOT = REPO_ROOT / "pipeline-results"
VIZ_ROOT = REPO_ROOT / "viz"
DATA_DIR = VIZ_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
assert RESULTS_ROOT.is_dir(), f"pipeline-results not found at {RESULTS_ROOT}"
print("repo root:", REPO_ROOT)
print("viz output dir:", VIZ_ROOT)

## 1. Point-cloud loading helpers

Parses the 3DGS-family `point_cloud.ply` schema (`x,y,z,f_dc_*,opacity,scale_*,rot_*`)
and COLMAP's plain `input.ply` (`x,y,z,red,green,blue`), each subsampled uniformly at
random for a responsive WebGL load (Three.js renders these as a single `THREE.Points`
draw call each, so tens to low hundreds of thousands of points is comfortably fast --
much higher than was practical to embed in the notebook directly).

In [ ]:
SH_C0 = 0.28209479177387814  # degree-0 SH basis constant -> [0,1] RGB
MAX_POINTS = 200_000

def load_gaussian_xyz_rgb(path, max_points=MAX_POINTS, seed=0):
    v = PlyData.read(str(path))["vertex"].data
    n = len(v)
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=min(max_points, n), replace=False)
    xyz = np.stack([v["x"][idx], v["y"][idx], v["z"][idx]], axis=1).astype(np.float32)
    dc = np.stack([v["f_dc_0"][idx], v["f_dc_1"][idx], v["f_dc_2"][idx]], axis=1).astype(np.float64)
    rgb01 = np.clip(0.5 + SH_C0 * dc, 0.0, 1.0)
    return xyz, (rgb01 * 255).astype(np.uint8), n

def load_colmap_xyz_rgb(path, max_points=MAX_POINTS, seed=0):
    v = PlyData.read(str(path))["vertex"].data
    n = len(v)
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=min(max_points, n), replace=False)
    xyz = np.stack([v["x"][idx], v["y"][idx], v["z"][idx]], axis=1).astype(np.float32)
    if "red" in v.dtype.names:
        rgb_u8 = np.stack([v["red"][idx], v["green"][idx], v["blue"][idx]], axis=1).astype(np.uint8)
    else:
        rgb_u8 = np.full((len(idx), 3), 180, dtype=np.uint8)
    return xyz, rgb_u8, n

def write_point_bin(name, xyz, rgb_u8):
    pos_path = DATA_DIR / f"{name}_pos.bin"
    col_path = DATA_DIR / f"{name}_col.bin"
    pos_path.write_bytes(xyz.astype(np.float32).tobytes())
    col_path.write_bytes(rgb_u8.astype(np.uint8).tobytes())
    return {"n": int(len(xyz)), "pos": pos_path.name, "col": col_path.name}

def robust_bbox(xyz, lo=2.0, hi=98.0):
    # Real COLMAP sparse clouds routinely contain a handful of far-flung points from
    # noisy feature matches -- a raw min/max bbox lets a few such outliers balloon the
    # box to many times the size of the actual scene content (measured: counter's real
    # bbox spans up to 50 units on one axis, vs ~12 units for its 1st-99th percentile
    # range). Percentile-based bounds are robust to this and are what the viewer uses
    # to auto-frame the camera on each dataset switch.
    lo_pt = np.percentile(xyz, lo, axis=0)
    hi_pt = np.percentile(xyz, hi, axis=0)
    return lo_pt, hi_pt

print("helpers ready")

## 2. Camera route: loading COLMAP poses from `cameras.json`

Each run in `pipeline-results` has its own `cameras.json` (a different scene means a
different physical camera set), so camera data is loaded and exported **per run** in
§4, not once globally.

`cameras.json` stores, per training image, a world-space `position` (the camera
center) and a 3x3 `rotation` matrix. This is the same JSON convention 3DGS's own
`camera_to_JSON` writes and many WebGL 3DGS viewers already consume directly: rotation
is camera-to-world, so the camera's forward (viewing) direction in world space is its
**third column** (OpenCV convention, +Z looks into the scene). Sanity-checked below
(and again per-run in §4) by solving for the least-squares point every camera ray
points closest to, and confirming it lands near that run's own reconstructed scene
centroid (not off in empty space, which would indicate a flipped/transposed
convention).

In [ ]:
def load_cameras(path):
    cams = json.load(open(path))
    pos = np.array([c["position"] for c in cams], dtype=np.float64)
    rot = np.array([c["rotation"] for c in cams], dtype=np.float64)  # [N,3,3]
    fwd = rot[:, :, 2]  # third column = forward (OpenCV convention, C2W rotation)
    fwd = fwd / np.linalg.norm(fwd, axis=1, keepdims=True)
    names = [c["img_name"] for c in cams]
    return pos, fwd, names

def ray_convergence_point(pos, fwd):
    # least-squares point minimizing summed squared perpendicular distance to every ray
    A = np.zeros((3, 3)); b = np.zeros(3)
    for p, d in zip(pos, fwd):
        Pj = np.eye(3) - np.outer(d, d)
        A += Pj
        b += Pj @ p
    return np.linalg.solve(A, b)

def write_camera_bin(name, pos, fwd, names=None):
    pos_path = DATA_DIR / f"{name}_cam_pos.bin"
    fwd_path = DATA_DIR / f"{name}_cam_fwd.bin"
    pos_path.write_bytes(pos.astype(np.float32).tobytes())
    fwd_path.write_bytes(fwd.astype(np.float32).tobytes())
    entry = {"n": int(len(pos)), "pos": pos_path.name, "fwd": fwd_path.name}
    if names is not None:
        entry["image_names"] = list(names)  # lets the viewer's registration-replay
                                              # feature (see run_colmap_and_trace.ipynb)
                                              # look up a camera mesh by image name
    return entry

# illustrative check on one run -- the same check runs for every discovered run in section 4
cam_pos, cam_fwd, cam_names = load_cameras(RESULTS_ROOT/"FastGS/counter_base/counter/cameras.json")
converge = ray_convergence_point(cam_pos, cam_fwd)
scene_centroid = load_colmap_xyz_rgb(RESULTS_ROOT/"FastGS/counter_base/counter/input.ply")[0].mean(axis=0)
print(f"{len(cam_pos)} cameras loaded for FastGS/counter_base/counter")
print(f"camera-ray convergence point: {converge.round(3)}")
print(f"scene point-cloud centroid:  {scene_centroid.round(3)}")
print(f"distance between them: {np.linalg.norm(converge - scene_centroid):.3f} "
      f"(small relative to scene extent -> forward-direction convention confirmed correct)")

## 3. The densification "gap": why a per-point classifier doesn't work, and what does

The natural first idea for "the gap between initial and final" is: for each *final*
Gaussian, find its nearest neighbor in the *initial* COLMAP cloud, and call it "new"
if that distance is large. Tried below on FastGS's `counter_big` (initial: 155,767
COLMAP points, final: 468,793 Gaussians) — even at a generous 3x the initial cloud's
own median point spacing, **55% of final Gaussians still count as "new."** This isn't
wrong, it's just not informative: 3DGS densification works by *splitting/cloning*
existing Gaussians into new ones offset by a small perturbation, so the majority of
final Gaussians are literally new points near an old one almost by construction,
regardless of whether densification did anything spatially interesting.

**What's actually informative** is a spatial *density* comparison: divide space into
voxels, and for each voxel compare how many initial points vs. final Gaussians fall
inside it. A voxel where density grew a lot is where densification concentrated new
detail; a voxel with only initial points and no surviving final Gaussians nearby is
a COLMAP point that didn't end up mattering. This is computed as `log2((final_count+1)
/ (initial_count+1))` per occupied voxel, colored on a diverging scale (blue = density
dropped, white = unchanged, red = density grew) -- this is the "densification gap"
layer in the viewer.

In [ ]:
# --- the naive per-point classifier, shown once to demonstrate why it's uninformative ---
init_xyz_demo, _, _ = load_colmap_xyz_rgb(RESULTS_ROOT/"FastGS/counter_base/counter/input.ply")
final_xyz_demo, _, _ = load_gaussian_xyz_rgb(RESULTS_ROOT/"FastGS/counter_big/counter_big/point_cloud/iteration_30000/point_cloud.ply")

tree = cKDTree(init_xyz_demo)
typical_spacing = np.median(tree.query(init_xyz_demo, k=2)[0][:, 1])
d_final, _ = tree.query(final_xyz_demo, k=1)
for mult in [1.0, 2.0, 3.0]:
    frac_new = (d_final > typical_spacing * mult).mean()
    print(f"  per-point 'new' @ {mult}x median spacing: {frac_new*100:.1f}% of final Gaussians -- not very discriminating")
print()

# --- the voxel density-ratio approach used in the viewer ---
def growth_voxel_centers(init_xyz, final_xyz, voxel_size=None, max_voxels=150_000, seed=0):
    if voxel_size is None:
        t = cKDTree(init_xyz)
        voxel_size = 4.0 * np.median(t.query(init_xyz, k=2)[0][:, 1])
    mins = init_xyz.min(axis=0)
    init_idx = np.floor((init_xyz - mins) / voxel_size).astype(np.int64)
    final_idx = np.floor((final_xyz - mins) / voxel_size).astype(np.int64)

    init_count, final_count = defaultdict(int), defaultdict(int)
    for row in map(tuple, init_idx):
        init_count[row] += 1
    for row in map(tuple, final_idx):
        final_count[row] += 1

    all_voxels = list(set(init_count) | set(final_count))
    print(f"  voxel_size={voxel_size:.4f}: {len(all_voxels)} occupied voxels "
          f"(initial-only={len(set(init_count)-set(final_count))}, "
          f"final-only/new growth={len(set(final_count)-set(init_count))})")

    rng = np.random.default_rng(seed)
    if len(all_voxels) > max_voxels:
        sel = rng.choice(len(all_voxels), size=max_voxels, replace=False)
        all_voxels = [all_voxels[i] for i in sel]

    centers = np.array([(np.array(v) + 0.5) * voxel_size + mins for v in all_voxels], dtype=np.float32)
    ratios = np.array([(final_count.get(v, 0) + 1.0) / (init_count.get(v, 0) + 1.0) for v in all_voxels])
    return centers, np.log2(ratios)

def log_ratio_to_rgb(log_ratio, clip=3.0):
    norm = mcolors.Normalize(vmin=-clip, vmax=clip)
    cmap = mpl.colormaps["RdBu_r"]
    rgba = cmap(norm(np.clip(log_ratio, -clip, clip)))
    return (rgba[:, :3] * 255).astype(np.uint8)

print("voxel-ratio approach on the same data:")
centers_demo, log_ratio_demo = growth_voxel_centers(init_xyz_demo, final_xyz_demo)
print(f"  log2(density ratio) range: [{log_ratio_demo.min():.2f}, {log_ratio_demo.max():.2f}], "
      f"median={np.median(log_ratio_demo):.2f}")

## 4. Discover every run in `pipeline-results` and export all of them

Rather than hand-picking one or two runs, this scans `pipeline-results/*/*/*` for every
directory that looks like a real training run (has a `point_cloud/iteration_*/` with a
saved `.ply`, plus the `input.ply` and `cameras.json` COLMAP wrote alongside it) and
exports **all** of them — `*_backup_<timestamp>` directories are skipped, and a run
folder with no `point_cloud/` at all (e.g. `SpecularGaussian/mip360/bonsai`, which only
has COLMAP setup files and was never trained) is skipped too. Each run uses its own
last saved iteration as "final" and its own `input.ply` as "initial", so the
densification-gap voxels and camera route are always computed against the correct
matching scene. The viewer's dataset dropdown is populated directly from whatever ends
up in `manifest.json`, so adding a new run later is just a matter of re-running this
cell.

In [ ]:
def discover_runs(results_root):
    """Find every real (non-backup) training run under pipeline-results that has a
    final .ply, its own input.ply, and its own cameras.json."""
    runs = {}
    for p in sorted(results_root.glob("*/*/*")):
        if not p.is_dir() or "backup" in p.name:
            continue
        pc_dir = p / "point_cloud"
        if not pc_dir.is_dir():
            continue
        iters = sorted([d for d in pc_dir.iterdir() if d.is_dir()],
                        key=lambda d: int(d.name.split("_")[-1]))
        if not iters:
            continue
        final_ply = iters[-1] / "point_cloud.ply"
        input_ply = p / "input.ply"
        cameras_json = p / "cameras.json"
        if not (final_ply.exists() and input_ply.exists() and cameras_json.exists()):
            continue

        method, scene_group, run_name = p.parts[-3], p.parts[-2], p.name
        if scene_group == run_name:
            key, label = f"{method.lower()}_{run_name}", f"{method} / {run_name}"
        else:
            key = f"{method.lower()}_{scene_group}_{run_name}"
            label = f"{method} / {scene_group}/{run_name}"
        runs[key] = {"label": label, "final_ply": final_ply, "input_ply": input_ply,
                     "cameras_json": cameras_json, "final_iter": iters[-1].name}
    return runs

RUNS = discover_runs(RESULTS_ROOT)
print(f"discovered {len(RUNS)} runs:")
for k, v in RUNS.items():
    print(f"  {k:35s} <- {v['label']}  ({v['final_iter']})")

In [ ]:
manifest = {"datasets": {}}

for key, cfg in RUNS.items():
    init_xyz, init_rgb, init_n = load_colmap_xyz_rgb(cfg["input_ply"])
    final_xyz, final_rgb, final_n = load_gaussian_xyz_rgb(cfg["final_ply"])

    initial_entry = write_point_bin(f"{key}_initial", init_xyz, init_rgb)
    final_entry = write_point_bin(f"{key}_final", final_xyz, final_rgb)

    centers, log_ratio = growth_voxel_centers(init_xyz, final_xyz)
    growth_entry = write_point_bin(f"{key}_growth", centers, log_ratio_to_rgb(log_ratio))

    # robust bbox from the FINAL cloud only -- it's the one the viewer actually shows by
    # default and is denser/more representative of true scene content than the sparse,
    # noisier init cloud
    bbox_lo, bbox_hi = robust_bbox(final_xyz)

    cam_pos, cam_fwd, cam_names = load_cameras(cfg["cameras_json"])
    converge = ray_convergence_point(cam_pos, cam_fwd)
    dist_to_centroid = np.linalg.norm(converge - init_xyz.mean(axis=0))
    camera_entry = write_camera_bin(key, cam_pos, cam_fwd, cam_names)

    manifest["datasets"][key] = {
        "label": cfg["label"],
        "initial": initial_entry,
        "final": final_entry,
        "growth": growth_entry,
        "cameras": camera_entry,
        "bbox_min": bbox_lo.tolist(),
        "bbox_max": bbox_hi.tolist(),
        "legend_html": (
            f"<b>{cfg['label']}</b><br>"
            f"initial: {init_n:,} COLMAP pts (showing {initial_entry['n']:,})<br>"
            f"final: {final_n:,} Gaussians @ {cfg['final_iter']} (showing {final_entry['n']:,})<br>"
            f"cameras: {camera_entry['n']}<br>"
            f"<span class='swatch' style='background:#2166ac'></span>density dropped &nbsp;"
            f"<span class='swatch' style='background:#f7f7f7;border:1px solid #999'></span>unchanged &nbsp;"
            f"<span class='swatch' style='background:#b2182b'></span>density grew"
        ),
    }
    print(f"{key}: init {init_n}->{initial_entry['n']}, final {final_n}->{final_entry['n']}, "
          f"growth voxels {growth_entry['n']}, cameras {camera_entry['n']} "
          f"(ray-convergence dist to centroid: {dist_to_centroid:.3f}, "
          f"robust bbox size: {(bbox_hi - bbox_lo).round(2)})")

# MERGE into any existing manifest.json rather than overwriting it wholesale: a companion
# notebook (run_colmap_and_trace.ipynb) adds its own live-COLMAP-run dataset entries to
# the same file, and re-running this cell shouldn't wipe those out.
manifest_path = DATA_DIR / "manifest.json"
if manifest_path.exists():
    existing = json.load(open(manifest_path))
    existing.setdefault("datasets", {}).update(manifest["datasets"])
    manifest = existing
with open(manifest_path, "w") as f:
    json.dump(manifest, f)
print(f"\nmanifest.json written -- {len(manifest['datasets'])} datasets total")

**Caveat noticed in the export output above**: the three Ref-NeRF runs (`toaster`,
`teapot`) show **zero** "final-only/new growth" voxels, unlike the real Mip-NeRF 360
`counter` runs. This isn't a bug — checking `input.ply`'s own statistics confirms why:
`counter`'s cloud has an irregular bounding box (45×24×50) with different spread per
axis, consistent with real COLMAP triangulation of an actual room; `toaster`'s cloud
has a near-perfectly cubic bounding box (2.6×2.6×2.6) with nearly identical std on
every axis (0.750/0.751/0.752) — the signature of a **uniform random initialization**
(typical for synthetic Blender/Ref-NeRF scenes that skip real feature-matching SfM).
A random cloud already densely fills the coarse voxel grid everywhere, so no voxel
ever looks "final-only." The densification-gap layer is most informative on the real
COLMAP-initialized scenes (`counter_*`) for this reason.

## 5. Write the WebGL viewer itself

The full viewer (HTML/CSS + a Three.js module script) is written out below so the notebook is fully self-contained and reproducible -- deleting viz/ and re-running regenerates everything, data and viewer alike.

In [ ]:
INDEX_HTML = r"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Gaussian Point-Cloud Viewer</title>
<style>
  html, body { margin:0; height:100%; overflow:hidden; font-family: -apple-system, Segoe UI, sans-serif; background:#111318; color:#e8e8ea; }
  #panel { position:absolute; top:12px; left:12px; background:rgba(18,18,22,0.88); padding:14px 16px; border-radius:10px; font-size:13px; width:280px; z-index:10; border:1px solid rgba(255,255,255,0.08); }
  #panel h3 { margin:0 0 10px 0; font-size:14px; font-weight:600; }
  #panel label { display:flex; align-items:center; gap:6px; margin:6px 0; cursor:pointer; }
  #panel select { width:100%; padding:4px; margin-bottom:8px; background:#22242b; color:#eee; border:1px solid #3a3d47; border-radius:4px; }
  #panel input[type=range] { width:100%; }
  #status { position:absolute; bottom:12px; left:12px; font-size:11px; color:#8a8d99; }
  #legend { margin-top:10px; font-size:11px; line-height:1.6; border-top:1px solid rgba(255,255,255,0.1); padding-top:8px; }
  .swatch { display:inline-block; width:10px; height:10px; margin-right:6px; border-radius:2px; vertical-align:middle; }
  .hint { font-size:10.5px; color:#8a8d99; margin-top:8px; line-height:1.5; }
  #replayBox { display:none; margin-top:10px; border-top:1px solid rgba(255,255,255,0.1); padding-top:8px; }
  #replayBox h4 { margin:0 0 6px 0; font-size:12.5px; font-weight:600; }
  #replayControls { display:flex; align-items:center; gap:6px; }
  #replayControls button { background:#2a2d36; color:#eee; border:1px solid #3a3d47; border-radius:4px; padding:4px 8px; cursor:pointer; font-size:12px; }
  #replayControls button:hover { background:#34384333; }
  #replayInfo { font-size:11px; color:#c7c9d1; margin-top:6px; line-height:1.5; }
</style>
</head>
<body>
<div id="panel">
  <h3>Gaussian Point-Cloud Viewer</h3>
  <select id="datasetSelect"></select>
  <label><input type="checkbox" id="toggleInitial" checked> Initial (COLMAP sparse)</label>
  <label><input type="checkbox" id="toggleFinal" checked> Final (trained Gaussians)</label>
  <label><input type="checkbox" id="toggleGrowth"> Densification gap (voxel density ratio)</label>
  <label><input type="checkbox" id="toggleCameras" checked> Camera route</label>
  <label>Point size <input type="range" id="pointSize" min="0.5" max="6" step="0.1" value="2"></label>
  <div id="legend"></div>
  <div id="replayBox">
    <h4>COLMAP registration replay</h4>
    <input type="range" id="replaySlider" min="0" max="0" step="1" value="0">
    <div id="replayControls">
      <button id="replayPlay">Play</button>
      <button id="replayReset">Reset</button>
      <label style="margin:0">Speed
        <input type="range" id="replaySpeed" min="1" max="10" step="1" value="4" style="width:70px">
      </label>
    </div>
    <div id="replayInfo"></div>
  </div>
  <div class="hint">Drag to orbit, scroll to zoom, right-drag to pan.</div>
</div>
<div id="status">Loading...</div>
<script type="importmap">
{ "imports": {
    "three": "https://unpkg.com/three@0.160.0/build/three.module.js",
    "three/addons/": "https://unpkg.com/three@0.160.0/examples/jsm/"
} }
</script>
<script type="module">
import * as THREE from "three";
import { OrbitControls } from "three/addons/controls/OrbitControls.js";

const statusEl = document.getElementById("status");

async function loadManifest() {
  const res = await fetch("data/manifest.json");
  return res.json();
}

async function loadBin(path, Ctor) {
  const res = await fetch(path);
  const buf = await res.arrayBuffer();
  return new Ctor(buf);
}

async function loadJSON(path) {
  const res = await fetch(path);
  return res.json();
}

async function loadPointSet(entry, baseDir) {
  const pos = await loadBin(`${baseDir}/${entry.pos}`, Float32Array);
  const col = await loadBin(`${baseDir}/${entry.col}`, Uint8Array);
  const geom = new THREE.BufferGeometry();
  geom.setAttribute("position", new THREE.BufferAttribute(pos, 3));
  geom.setAttribute("color", new THREE.BufferAttribute(col, 3, true));
  return geom;
}

let scene, camera, renderer, controls;
let pointsInitial, pointsFinal, pointsGrowth, cameraGroup;
let manifest;

// registration replay state
let regOrder = null;        // ordered list of {step, image_name, elapsed_s, num_registered_so_far, num_points3D}
let regNameToMesh = null;   // image_name -> camera frustum mesh, for show/hide during replay
let regPlaying = false;
let regPlayTimer = null;

function initScene() {
  scene = new THREE.Scene();
  scene.background = new THREE.Color(0x111318);
  camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 0.005, 1000);
  camera.position.set(2.2, 2.2, 2.2);
  renderer = new THREE.WebGLRenderer({ antialias: true, preserveDrawingBuffer: true });
  renderer.setSize(window.innerWidth, window.innerHeight);
  document.body.appendChild(renderer.domElement);
  controls = new OrbitControls(camera, renderer.domElement);
  controls.enableDamping = true;

  scene.add(new THREE.AxesHelper(0.5));

  window.addEventListener("resize", () => {
    camera.aspect = window.innerWidth / window.innerHeight;
    camera.updateProjectionMatrix();
    renderer.setSize(window.innerWidth, window.innerHeight);
  });
}

function makePointsMaterial(size) {
  return new THREE.PointsMaterial({ size, vertexColors: true, sizeAttenuation: false });
}

function disposePoints(obj) {
  if (!obj) return;
  scene.remove(obj);
  obj.geometry.dispose();
  obj.material.dispose();
}

function disposeGroup(group) {
  if (!group) return;
  scene.remove(group);
  group.traverse(child => {
    if (child.geometry) child.geometry.dispose();
    if (child.material) child.material.dispose();
  });
}

// each dataset can be a totally different scene at a totally different physical scale
// (a tabletop teapot vs. a whole kitchen counter), so re-frame the camera/orbit target
// on every dataset switch instead of leaving the view pointed wherever it was. Uses the
// PRECOMPUTED, PERCENTILE-based bbox from the manifest (ds.bbox_min/bbox_max), not a raw
// Three.js geometry bounding box -- real COLMAP clouds routinely have a handful of
// far-flung outlier points from noisy feature matches that would otherwise balloon a
// raw min/max box to many times the size of the actual scene content.
function frameToBounds(bboxMin, bboxMax) {
  const min = new THREE.Vector3(...bboxMin);
  const max = new THREE.Vector3(...bboxMax);
  const size = new THREE.Vector3().subVectors(max, min);
  const center = new THREE.Vector3().addVectors(min, max).multiplyScalar(0.5);
  const maxDim = Math.max(size.x, size.y, size.z) || 1;
  const dist = maxDim * 1.3;
  camera.position.set(center.x + dist, center.y + dist * 0.8, center.z + dist);
  camera.near = Math.max(0.001, maxDim / 1000);
  camera.far = dist * 8 + maxDim * 4;
  camera.updateProjectionMatrix();
  controls.target.copy(center);
  controls.update();
}

async function loadCamerasFor(ds, baseDir) {
  if (!ds.cameras) return { group: null, nameToMesh: null };
  const pos = await loadBin(`${baseDir}/${ds.cameras.pos}`, Float32Array);
  const fwd = await loadBin(`${baseDir}/${ds.cameras.fwd}`, Float32Array);
  const names = ds.cameras.image_names || null;
  const group = new THREE.Group();
  const nameToMesh = names ? {} : null;

  const n = pos.length / 3;
  const linePts = [];
  for (let i = 0; i < n; i++) linePts.push(new THREE.Vector3(pos[i*3], pos[i*3+1], pos[i*3+2]));
  const lineGeom = new THREE.BufferGeometry().setFromPoints(linePts);
  group.add(new THREE.Line(lineGeom, new THREE.LineBasicMaterial({ color: 0xffcc33 })));

  const span = new THREE.Box3().setFromPoints(linePts).getSize(new THREE.Vector3());
  const frustumLen = Math.max(0.01, 0.03 * Math.max(span.x, span.y, span.z));
  const coneGeom = new THREE.ConeGeometry(frustumLen * 0.4, frustumLen, 4);
  coneGeom.rotateX(Math.PI / 2); // default apex +Y -> +Z, so mesh.quaternion aligns +Z with forward
  const coneMat = new THREE.MeshBasicMaterial({ color: 0x4da6ff, wireframe: true, transparent: true, opacity: 0.85 });

  const zAxis = new THREE.Vector3(0, 0, 1);
  for (let i = 0; i < n; i++) {
    const p = new THREE.Vector3(pos[i*3], pos[i*3+1], pos[i*3+2]);
    const d = new THREE.Vector3(fwd[i*3], fwd[i*3+1], fwd[i*3+2]).normalize();
    const mesh = new THREE.Mesh(coneGeom, coneMat);
    mesh.position.copy(p);
    mesh.quaternion.setFromUnitVectors(zAxis, d);
    group.add(mesh);
    if (names && names[i] !== undefined) nameToMesh[names[i]] = mesh;
  }
  return { group, nameToMesh };
}

function stopReplay() {
  regPlaying = false;
  document.getElementById("replayPlay").textContent = "Play";
  if (regPlayTimer) { clearInterval(regPlayTimer); regPlayTimer = null; }
}

// Reveal only the cameras registered up to (and including) regOrder[stepIdx-1] -- this
// replays COLMAP's own incremental registration order captured during the run (see
// tools/run_colmap_and_trace.ipynb), not just an arbitrary index/name ordering.
function applyRegistrationStep(stepIdx) {
  if (!regOrder || !regNameToMesh) return;
  const visibleNames = new Set(regOrder.slice(0, stepIdx).map(e => e.image_name));
  for (const [name, mesh] of Object.entries(regNameToMesh)) {
    mesh.visible = visibleNames.has(name);
  }
  const info = document.getElementById("replayInfo");
  if (stepIdx === 0) {
    info.textContent = "Step 0 / " + regOrder.length + " -- no cameras registered yet";
  } else {
    const e = regOrder[stepIdx - 1];
    info.textContent = `Step ${stepIdx} / ${regOrder.length} -- registered "${e.image_name}" `
      + `@ t=${e.elapsed_s.toFixed(1)}s (${e.num_registered_so_far} cams, ${e.num_points3D.toLocaleString()} points3D so far)`;
  }
}

function setupReplayUI() {
  const box = document.getElementById("replayBox");
  const slider = document.getElementById("replaySlider");
  const playBtn = document.getElementById("replayPlay");
  const resetBtn = document.getElementById("replayReset");
  const speed = document.getElementById("replaySpeed");

  if (!regOrder || !regNameToMesh) {
    box.style.display = "none";
    return;
  }
  box.style.display = "block";
  slider.max = String(regOrder.length);
  slider.value = String(regOrder.length); // start fully revealed, matching the non-replay default
  applyRegistrationStep(regOrder.length);

  slider.oninput = () => { stopReplay(); applyRegistrationStep(parseInt(slider.value, 10)); };
  resetBtn.onclick = () => { stopReplay(); slider.value = "0"; applyRegistrationStep(0); };
  playBtn.onclick = () => {
    if (regPlaying) { stopReplay(); return; }
    regPlaying = true;
    playBtn.textContent = "Pause";
    if (parseInt(slider.value, 10) >= regOrder.length) slider.value = "0";
    regPlayTimer = setInterval(() => {
      let v = parseInt(slider.value, 10) + 1;
      if (v > regOrder.length) { stopReplay(); return; }
      slider.value = String(v);
      applyRegistrationStep(v);
    }, Math.max(30, 400 / parseInt(speed.value, 10)));
  };
}

async function loadDataset(key) {
  statusEl.textContent = "Loading dataset...";
  const ds = manifest.datasets[key];
  const baseDir = "data";

  stopReplay();
  disposePoints(pointsInitial); pointsInitial = null;
  disposePoints(pointsFinal); pointsFinal = null;
  disposePoints(pointsGrowth); pointsGrowth = null;
  disposeGroup(cameraGroup); cameraGroup = null;
  regOrder = null; regNameToMesh = null;

  const size = parseFloat(document.getElementById("pointSize").value);

  if (ds.initial) {
    const g = await loadPointSet(ds.initial, baseDir);
    pointsInitial = new THREE.Points(g, makePointsMaterial(size));
    pointsInitial.visible = document.getElementById("toggleInitial").checked;
    scene.add(pointsInitial);
  }
  if (ds.final) {
    const g = await loadPointSet(ds.final, baseDir);
    pointsFinal = new THREE.Points(g, makePointsMaterial(size));
    pointsFinal.visible = document.getElementById("toggleFinal").checked;
    scene.add(pointsFinal);
  }
  if (ds.growth) {
    const g = await loadPointSet(ds.growth, baseDir);
    pointsGrowth = new THREE.Points(g, makePointsMaterial(size * 1.6));
    pointsGrowth.visible = document.getElementById("toggleGrowth").checked;
    scene.add(pointsGrowth);
  }
  if (ds.bbox_min && ds.bbox_max) frameToBounds(ds.bbox_min, ds.bbox_max);

  const camResult = await loadCamerasFor(ds, baseDir);
  cameraGroup = camResult.group;
  regNameToMesh = camResult.nameToMesh;
  if (cameraGroup) {
    cameraGroup.visible = document.getElementById("toggleCameras").checked;
    scene.add(cameraGroup);
  }

  if (ds.registration_order) {
    regOrder = await loadJSON(`${baseDir}/${ds.registration_order.path}`);
  }
  setupReplayUI();

  document.getElementById("legend").innerHTML = ds.legend_html || "";
  statusEl.textContent = `Loaded: ${ds.label}`;
}

function setupUI() {
  const sel = document.getElementById("datasetSelect");
  for (const key in manifest.datasets) {
    const opt = document.createElement("option");
    opt.value = key;
    opt.textContent = manifest.datasets[key].label;
    sel.appendChild(opt);
  }
  sel.addEventListener("change", () => loadDataset(sel.value));

  document.getElementById("toggleInitial").addEventListener("change", e => { if (pointsInitial) pointsInitial.visible = e.target.checked; });
  document.getElementById("toggleFinal").addEventListener("change", e => { if (pointsFinal) pointsFinal.visible = e.target.checked; });
  document.getElementById("toggleGrowth").addEventListener("change", e => { if (pointsGrowth) pointsGrowth.visible = e.target.checked; });
  document.getElementById("toggleCameras").addEventListener("change", e => { if (cameraGroup) cameraGroup.visible = e.target.checked; });
  document.getElementById("pointSize").addEventListener("input", e => {
    const s = parseFloat(e.target.value);
    if (pointsInitial) pointsInitial.material.size = s;
    if (pointsFinal) pointsFinal.material.size = s;
    if (pointsGrowth) pointsGrowth.material.size = s * 1.6;
  });
}

function animate() {
  requestAnimationFrame(animate);
  controls.update();
  renderer.render(scene, camera);
}

async function main() {
  initScene();
  manifest = await loadManifest();
  setupUI();
  const firstKey = Object.keys(manifest.datasets)[0];
  document.getElementById("datasetSelect").value = firstKey;
  await loadDataset(firstKey);
  animate();
}
main().catch(err => { statusEl.textContent = "Error: " + err.message; console.error(err); });
</script>
</body>
</html>
"""

(VIZ_ROOT / "index.html").write_text(INDEX_HTML, encoding="utf-8")
print("viewer HTML written:", VIZ_ROOT / "index.html", f"({len(INDEX_HTML)} chars)")

## 6. Start the local server

Plain `http.server` (Python standard library, no extra dependency), run as a background
subprocess so the notebook keeps working while it serves. Tries a small range of ports
in case 8000 is already taken.

In [ ]:
import subprocess, sys, time, urllib.request

def start_server(directory, ports=range(8000, 8010)):
    for port in ports:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}", timeout=0.2)
            continue  # something already answers on this port, try the next one
        except Exception:
            pass
        proc = subprocess.Popen(
            [sys.executable, "-m", "http.server", str(port), "--directory", str(directory)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        time.sleep(0.6)
        if proc.poll() is None:  # still running -> bound the port successfully
            return proc, port
    raise RuntimeError(f"could not bind any port in {ports}")

_server_proc, _server_port = start_server(VIZ_ROOT)
url = f"http://localhost:{_server_port}/"
print(f"Serving {VIZ_ROOT} at {url}")
print("Open that link in a browser. Run the cell below when you're done to stop the server.")

## 7. Stop the server (run when done viewing)

In [ ]:
# _server_proc.terminate()
# _server_proc.wait(timeout=5)
# print("server stopped")

## Summary

- §4 auto-discovers every real training run under `pipeline-results` (currently 7:
  FastGS on `counter_base`, `counter_big`, `toaster`; Spec-Gaussian on
  `counter_images_4`, `counter_images_8`, `refnerf/teapot`, `refnerf/toaster`) rather
  than hard-coding one or two — the viewer's dropdown lists all of them, built directly
  from `manifest.json`.
- Each run's initial cloud, final Gaussians, densification-gap voxels, and camera route
  are all computed against **that run's own scene**, never mixed across runs.
- Switching datasets re-frames the camera/orbit target to the new scene's own bounding
  box, since different runs can be wildly different physical scales (a tabletop teapot
  vs. a whole kitchen counter) — without this the view could end up pointed at empty
  space after a switch.
- The camera forward-direction convention is sanity-checked per run (ray-convergence
  distance to that scene's own point-cloud centroid), printed during export in §4.
- Adding a new run later requires no code changes: drop it in `pipeline-results` with
  the same `point_cloud/iteration_*/point_cloud.ply` + `input.ply` + `cameras.json`
  layout and re-run §4 (and §6 to restart the server, if it isn't already running).